# MedSafe: source-topic training experiment

This notebook reproduces the **new topic classifier**, not the original three-category model. It learns which discussion topics appear in source guidance from the interaction description. It does not write medical advice, determine severity, or verify safety.

The app retrieves DDInter records separately. Its questions use topics mentioned in the selected record; the classifier can optionally highlight those existing questions. A mention may be conditional or background, so a clinician must interpret it.

The frozen dataset has 886 examples in 259 similar-text groups: 581 training, 120 validation, 185 test. Targets were assigned using documented text rules (weak supervision), not clinician review. The model was trained locally for this export; running this notebook reproduces training in your own Colab session.

**Current result:** macro F1 0.105 for the classifier, versus 0.369 for the keyword baseline. This is not a successful replacement for the baseline. Model highlighting is off by default. Do not tune on this test set or claim clinical accuracy.


## 1. Install the matching packages
Use a fresh Colab runtime. If Colab asks for a restart after installation, restart before continuing. The export expects these exact versions.

In [ ]:
%pip install -q numpy==2.1.3 pandas==2.2.3 scikit-learn==1.6.1 joblib==1.6.0


## 2. Upload these three files from your GitHub checkout
- `MedSafe/training/source_topic_examples.json`
- `MedSafe/training/train_source_topics.py`
- `MedSafe/backend/source_topics.py`

No Excel or editing hundreds of rows is needed. Each example keeps its source URL, source hash, label evidence, and group. Keep this frozen dataset unchanged when reproducing the reported results.


In [ ]:
from google.colab import files
from pathlib import Path
import hashlib, json
uploaded = files.upload()
required = {'source_topic_examples.json', 'train_source_topics.py', 'source_topics.py'}
assert required <= set(uploaded), f'Missing: {required - set(uploaded)}'
expected = 'b4004af2eaf2ffdfc6336186f7cd8529179ed038e7151ea1d7b23cb4ac607a79'
assert hashlib.sha256(Path('source_topic_examples.json').read_bytes()).hexdigest() == expected, 'This is a different dataset snapshot.'
examples = json.loads(Path('source_topic_examples.json').read_text())
print('Examples:', len(examples['examples']))
print('Labels:', examples['topics'])
print(examples['label_method'])


## 3. Train and evaluate
The script groups similar interaction descriptions together to reduce near-duplicate leakage. Only training data fits the vocabulary and classifier. Validation chooses C from 0.1, 1, and 10. The test set evaluates the chosen model once. A majority baseline and a keyword baseline are reported alongside it.

Inputs exclude management paragraphs, URLs, and target labels. They can still contain discussion-topic words, because those also occur naturally in interaction descriptions. Automated source labels are imperfect; same-source group splitting is not an external clinical test.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'train_source_topics.py', '--examples', 'source_topic_examples.json', '--output', 'topic_export'], check=True)


## 4. Inspect the results honestly
Some topics have zero test support; macro F1 includes all trained topics, and those zero-support topics score zero. Review per-topic precision, recall, and support rather than reporting a single number as medical accuracy. Magnesium lacked training support and is not a model target; the app can still show a magnesium question when mentioned by the source.

In [ ]:
import pandas as pd
root = Path('topic_export')
evaluation = json.loads((root / 'source_topic_evaluation.json').read_text())
metadata = json.loads((root / 'source_topic_metadata.json').read_text())
display(pd.DataFrame([{'method': name, 'macro_f1': report['macro_f1']} for name, report in evaluation.items()]))
display(pd.DataFrame(evaluation['trained_topic_selector']['per_topic']).T)
print('Rows:', metadata['split_rows'])
print('Groups:', metadata['split_groups'])
print('Untrained topics:', metadata['untrained_topics'])


## 5. Download the five export files
Keep the existing original and coverage models. Put `source_topic_selector.joblib` and `source_topic_metadata.json` in `MedSafe/backend/models/`. Put the evaluation, split, and held-out JSON files in `MedSafe/training/results/`. Keep `source_topic_examples.json` in `MedSafe/training/` because startup verifies its hash.

Run each download separately if your browser blocks multiple downloads. These files are not packaged in a ZIP.


In [ ]:
files.download(str(root / 'source_topic_selector.joblib'))


In [ ]:
files.download(str(root / 'source_topic_metadata.json'))


In [ ]:
files.download(str(root / 'source_topic_evaluation.json'))


In [ ]:
files.download(str(root / 'source_topic_split.json'))


In [ ]:
files.download(str(root / 'source_topic_heldout.json'))


## Next research step
Have qualified reviewers label independent source families, especially poorly supported topics. Freeze a new external test set before model changes. Compare improved models using training and validation only, then evaluate the frozen test set. More rows copied from the same explanation do not necessarily add new training information.

Source-derived examples retain DDInter attribution and its CC BY-NC-SA 4.0 terms. See `MedSafe/backend/data/SOURCE.md`.
